# Hệ thống AI gợi ý ẩm thực và tối ưu hóa ngân sách

**Môn học:** Nhập môn Trí tuệ Nhân tạo (CO3061)  
**Học kỳ:** II — Năm học 2025–2026  
**Trường:** Đại học Bách Khoa, ĐHQG-HCM  
**GVHD:** TS. Trương Vĩnh Lân  

| Họ và tên | MSSV | Email |
|-----------|------|-------|
| Đinh Đoàn Vy | 2353350 | vy.dinhdoan2005@hcmut.edu.vn |
| Trần Thiên Lộc | 2352715 | loc.tranthien3905@hcmut.edu.vn |
| Nguyễn Trần Gia Bảo | 2352103 | bao.nguyentrangia@hcmut.edu.vn |
| Dương Lê Nhật Duy | 2352171 | duy.duong250405@hcmut.edu.vn |

**Repository:** https://github.com/dvydinh/yumify  
**Báo cáo:** `reports/report.pdf`

---
## 1. Cài đặt môi trường

Clone mã nguồn từ GitHub và cài đặt thư viện. Không mount Google Drive.

In [ ]:
import os, sys

REPO_URL = "https://github.com/dvydinh/yumify.git"
PROJECT_DIR = "/content/yumify"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin main

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

!pip install -q numpy requests

print(f"Working directory: {os.getcwd()}")

---
## 2. Nạp dữ liệu và trích xuất đặc trưng TF-IDF

Ưu tiên tải dataset Kaggle Food.com (hàng nghìn recipes). Nếu không khả dụng, fallback về `recipes.json` (25 recipes cục bộ).

In [ ]:
import json
import numpy as np
from collections import Counter
from features.feature_extractor import download_recipe_dataset, RecipeFeatureExtractor

data_dir = os.path.join(PROJECT_DIR, "data")

# Nạp dataset: Kaggle (primary) -> recipes.json (fallback)
recipes = download_recipe_dataset(data_dir)

with open(os.path.join(data_dir, "ingredients.json"), "r", encoding="utf-8") as f:
    ingredients_db = json.load(f)

print(f"\nRecipes: {len(recipes)} (source: {'kaggle' if len(recipes) > 50 else 'recipes.json'})")
print(f"Ingredients DB: {len(ingredients_db)}")

cuisine_dist = Counter(r.get("cuisine", "N/A") for r in recipes)
print(f"\nPhan bo cuisine:")
for c, n in cuisine_dist.most_common(10):
    print(f"  {c}: {n}")

# Cost sanity check
costs = [r.get("cost", 0) for r in recipes if r.get("cost", 0) > 0]
if costs:
    print(f"\nCost range: ${min(costs):.2f} - ${max(costs):.2f}, mean=${sum(costs)/len(costs):.2f}")
else:
    print("\nWarning: no recipes have cost data")

# TF-IDF
extractor = RecipeFeatureExtractor()
extractor.fit(recipes)
extractor.save_embeddings(os.path.join(data_dir, "tfidf_features"))

print(f"\nTF-IDF: {len(extractor.vocabulary)} terms, {len(extractor.tfidf_matrix)} documents")

---
## 3. Khởi tạo các module AI

In [ ]:
from modules.nlp_parser import EnglishNLPParser
from modules.csp_solver import IngredientCSPSolver
from modules.search_engine import AStarMealPlanner
from modules.knowledge_base import DietaryKnowledgeBase
from modules.bayes_risk import BayesianRecipeEvaluator
from modules.ml_classifier import CuisineNaiveBayesClassifier

nlp_parser = EnglishNLPParser()
csp_solver = IngredientCSPSolver()
search_engine = AStarMealPlanner()
kb = DietaryKnowledgeBase()
bayes_eval = BayesianRecipeEvaluator()

# Train ML classifier on the full dataset (not just recipes.json)
if len(recipes) > len(nlp_parser.ml_classifier.class_counts) * 5:
    print(f"Re-training ML classifier on {len(recipes)} recipes...")
    nlp_parser.ml_classifier.train(recipes)
    print(f"Trained: {len(nlp_parser.ml_classifier.class_counts)} cuisines, {len(nlp_parser.ml_classifier.vocab)} vocab")

print("Modules loaded.")

---
## 4. Pipeline tổng hợp

Chạy toàn bộ pipeline trên một câu input mẫu:

```
NLP (NER) -> ML Naive Bayes -> Knowledge Base -> Filter -> Bayesian Risk -> Rank
```

In [ ]:
test_input = "I want to cook something with beef and noodle for $10, I have stomachache"

print("Input:", test_input)
print()

# Step 1: NLP
parsed = nlp_parser.parse(test_input)
print("--- NLP Parser ---")
print(f"  Budget:      ${parsed.budget}")
print(f"  Ingredients: {parsed.ingredients}")
print(f"  Health:      {parsed.health_conditions}")
print(f"  Cuisine:     {parsed.target_cuisine} ({parsed.cuisine_method}, {parsed.cuisine_confidence:.4f})")
print(f"  Confidence:  {parsed.confidence:.2f}")

# Step 2: Knowledge Base (Forward Chaining)
kb_result = kb.evaluate(health_conditions=parsed.health_conditions)
print("\n--- Knowledge Base ---")
print(f"  Fired rules:  {kb_result.fired_rules}")
print(f"  Excluded ings: {kb_result.excluded_ingredients}")
print(f"  Excluded tags: {kb_result.excluded_tags}")
for w in kb_result.warnings:
    print(f"  Warning: {w}")

# Step 3: Filter
excluded_ings = set(kb_result.excluded_ingredients)
excluded_tags = set(kb_result.excluded_tags)
filtered = [
    r for r in recipes
    if not {i.get("name", "").lower() for i in r.get("ingredients", [])}.intersection(excluded_ings)
    and not {t.lower() for t in r.get("tags", [])}.intersection(excluded_tags)
]
print(f"\n--- Filtering ---")
print(f"  {len(recipes)} total -> {len(filtered)} after KB filtering")

# Step 4: Bayesian evaluation
evaluated = []
for r in filtered[:20]:
    ev = bayes_eval.evaluate(r, user_conditions=parsed.health_conditions)
    evaluated.append((r, ev))
evaluated.sort(key=lambda x: x[1].preference_score, reverse=True)

print(f"\n--- Bayesian Risk Assessment ---")
for i, (r, ev) in enumerate(evaluated[:5], 1):
    name = r.get("name", "N/A")
    cost = r.get("cost", 0)
    print(f"  {i}. {name:35s}  P(Like)={ev.preference_score:.3f}  Risk={ev.risk_score:.3f}  ${cost:.2f}")

# Step 5: ML cuisine predictions
top_k = nlp_parser.ml_classifier.predict_top_k(parsed.ingredients, k=3)
print(f"\n--- ML Cuisine Prediction ---")
for cuisine, conf in top_k:
    print(f"  {cuisine:15s} {conf:.2%}")

---
## 5. Kiểm thử từng module

### 5.1 Naive Bayes Classifier

In [ ]:
classifier = CuisineNaiveBayesClassifier()
classifier.train(recipes)
print(f"Trained on {sum(classifier.class_counts.values())} recipes, {len(classifier.class_counts)} classes, {len(classifier.vocab)} vocab")

test_cases = [
    (["olive oil", "tomato sauce", "pasta"], "Italian"),
    (["salmon", "sushi rice", "wasabi"], "Japanese"),
    (["beef", "fish sauce", "lemongrass"], "Vietnamese"),
]
for ings, expected in test_cases:
    pred, conf = classifier.predict_with_confidence(ings)
    print(f"  {str(ings):50s} -> {str(pred):15s} ({conf:.4f})")

### 5.2 CSP Solver

In [ ]:
test_recipe = recipes[0]
csp_result = csp_solver.solve(
    recipe=test_recipe,
    ingredients_db=ingredients_db,
    budget=10.0,
    calorie_range=(200, 800)
)
print(f"Recipe:     {test_recipe.get('name')}")
print(f"Success:    {csp_result.success}")
print(f"Cost:       ${csp_result.total_cost:.2f}")
print(f"Calories:   {csp_result.total_calories:.0f}")
print(f"Backtracks: {csp_result.backtracks}")

### 5.3 A* Search

In [ ]:
# Lay 8 recipes co cost > 0 de A* co the toi uu
candidates = [r for r in recipes if r.get("cost", 0) > 0][:8]
if not candidates:
    candidates = recipes[:8]

search_result = search_engine.search(
    candidate_recipes=candidates,
    budget=50.0,
    days=3,
    calorie_range_per_day=(200, 900)
)
print(f"Success:        {search_result.success}")
print(f"Nodes explored: {search_result.nodes_explored}")
if search_result.success:
    print(f"Total cost:     ${search_result.total_cost:.2f}")
    print(f"Total calories: {search_result.total_calories:.0f}")
    for i, r in enumerate(search_result.planned_recipes, 1):
        name = r.get('name', 'N/A')
        cost = r.get('cost', 0)
        cal = r.get('calories', 0)
        print(f"  Day {i}: {name} (${cost:.2f}, {cal} cal)")

### 5.4 Bayesian Risk — Extended Noisy-OR

In [ ]:
spicy_recipe = {
    "name": "Spicy Shrimp Oil Pasta",
    "ingredients": [
        {"name": "shrimp", "quantity": 150},
        {"name": "chili", "quantity": 10},
        {"name": "olive oil", "quantity": 30},
        {"name": "pasta", "quantity": 200},
    ],
    "tags": ["spicy", "seafood", "italian"]
}

risk, factors = bayes_eval.assess_risk(spicy_recipe, ["stomachache"])
print(f"Recipe: {spicy_recipe['name']}")
print(f"P(Risk) = {risk:.4f}")
for f in factors:
    factor_name = f.get('yếu_tố', '?')
    prob = f.get('xác_suất', '?')
    desc = f.get('mô_tả', '')
    print(f"  {factor_name:20s} P={prob}  {desc}")

mild = {
    "name": "Steamed vegetables",
    "ingredients": [{"name": "spinach"}, {"name": "carrot"}],
    "tags": ["healthy"]
}
mild_risk, _ = bayes_eval.assess_risk(mild)
print(f"\nComparison: {spicy_recipe['name']} risk={risk:.4f} vs {mild['name']} risk={mild_risk:.4f}")

---
## 6. Truy vấn đa dạng

In [ ]:
def analyze(text):
    parsed = nlp_parser.parse(text)
    result = kb.evaluate(health_conditions=parsed.health_conditions)
    excluded = set(result.excluded_ingredients)
    filtered = [
        r for r in recipes
        if not {i.get("name", "").lower() for i in r.get("ingredients", [])}.intersection(excluded)
    ]
    top = nlp_parser.ml_classifier.predict_top_k(parsed.ingredients, k=3)
    preds = ", ".join(f"{c} ({p:.0%})" for c, p in top)

    print(f"  Input:    {text}")
    print(f"  Cuisine:  {parsed.target_cuisine} ({parsed.cuisine_method})")
    print(f"  Ings:     {parsed.ingredients}")
    print(f"  Health:   {parsed.health_conditions}")
    print(f"  Recipes:  {len(filtered)}/{len(recipes)}")
    print(f"  ML top-3: {preds}")
    print()


queries = [
    "I want pasta with olive oil and tomato sauce for $8",
    "Sushi with salmon and rice, budget $15",
    "Something with beef and chili, I have diabetes",
    "Chicken soup with ginger, no spicy, I have gout",
    "Tofu and mushroom stir fry, vegetarian",
]

for q in queries:
    analyze(q)

---
## 7. Tìm kiếm tương tự (TF-IDF cosine similarity)

In [ ]:
search_queries = ["pasta cheese Italian", "sushi salmon Japanese", "spicy beef noodle soup"]

for q in search_queries:
    matches = extractor.find_similar(q, top_k=3)
    print(f"Query: '{q}'")
    for m in matches:
        name = m.recipe.get("name", "N/A")
        print(f"  {m.rank}. {name} (sim={m.similarity:.3f})")
    print()